In [ ]:
import sys
import pandas as pd
sys.path.append('..')
from src.utils import get_device, load_feature_file, set_seed, train_clustering

set_seed(42)
device = get_device()
print(f"[INFO] Thiết bị: {device}")

# Sử dụng lại file đặc trưng STL-10 đã lưu từ File 1
X, y = load_feature_file('../data/stl10_moco_features.npy', '../data/stl10_labels.npy', device=device)
print(f"[INFO] Load thành công đặc trưng STL-10: X={X.shape}, y={y.shape}")

[INFO] Thiết bị: cpu
[INFO] Load thành công đặc trưng STL-10: X=torch.Size([8000, 2048]), y=torch.Size([8000])


In [2]:
def run_ablation(name, enable_split, enable_merge):
    k_star, acc, nmi, ari = train_clustering(
        features=X, labels=y, k_max=20, device=device, epochs=50, lr=1e-3,
        lambda_param=2.0, gamma=0.1, warmup_epochs=20,
        enable_split=enable_split, enable_merge=enable_merge
    )
    return {
        'Cấu hình': name, 
        'Split (Tách)': 'Bật' if enable_split else 'Tắt', 
        'Merge (Gộp)': 'Bật' if enable_merge else 'Tắt', 
        'K*': int(k_star), 
        'ACC(%)': round(acc*100, 2)
    }

print("\n--- BẮT ĐẦU CHẠY ABLATION STUDY ---")
ablation_results = [
    run_ablation('Full PnP', True, True),
    run_ablation('Chỉ Merge', False, True),
    run_ablation('Chỉ Split', True, False),
    run_ablation('Baseline', False, False),
]

display(pd.DataFrame(ablation_results))


--- BẮT ĐẦU CHẠY ABLATION STUDY ---


,Cấu hình,Split (Tách),Merge (Gộp),K*,ACC(%)
0,Full PnP,Bật,Bật,12,71.80
1,Chỉ Merge,Tắt,Bật,11,73.69
2,Chỉ Split,Bật,Tắt,20,46.14
3,Baseline,Tắt,Tắt,20,40.15
